# Word Mover’s Distance(WMD)

NLP연구쪽에서 문서 유사도 구하는 방법으로 잘 쓰인다고 함. (논문 연구 결과 다른 방법론보다 성능이 좋다고 함)\
기본적으로 word2vec을 베이스로 하고, word2vec을 사용해서 word간의 euclide distance를 구함!

- 피험자 1명, seed 단어 1개

피험자가 대답한 30개의 연속적인 단어 그룹 - target word(money, friend, family)의 유사도를 기반으로 wmdistance를 잰다.

--> 그럼 3개의 distance값이 나옴. 그 중 가장 값이 작은 것이 거리가 가깝다. 즉, target word와 유사하다.

wmd 설명 참조
- https://sy-programmingstudy.tistory.com/14
- https://www.youtube.com/watch?v=zFnrq5SmBdg

In [1]:
import os
import platform

import pandas as pd
import numpy as np
from gensim.models import KeyedVectors

In [2]:
os_system = platform.system() # 맥북은 Darwin, 윈도우는 Windows

# 현재 프로젝트 폴더 위치 지정. os.getcwd()는 지금 코드 실행하는 현 위치를 출력해줍니다.
pilot2_dir = os.getcwd()
model_path = '\\..\\pretrained\\GoogleNews-vectors-negative300.bin' if os_system == 'Windows' else '/../pretrained/GoogleNews-vectors-negative300.bin'

# data/processed 폴더 위치 지정
processed_data_dir = pilot2_dir + ('\\data\\processed\\' if os_system == 'Windows' else '/data/processed/')

In [3]:
# word2vec model 로딩
word2vec_model = KeyedVectors.load_word2vec_format(pilot2_dir + model_path, binary=True)

In [4]:
# 테이블 읽어오기
pilot_data = pd.read_csv(processed_data_dir + 'merged_data.csv', keep_default_na=False)
pilot_data[0:3]

,Prolific_ID,subject,key1,key2,key3,key4,key5,key6,key7,key8,...,friend21,friend22,friend23,friend24,friend25,friend26,friend27,friend28,friend29,friend30
0,5d53bffa147a7d00015aae5a,1,Door,Gate,Outside,Grass,Itchy,Rash,Chicken pox,Shingles,...,Burned down,Scary,Movies,Knocked up,Tinsletown,Christmas tree,Holiday,Decorate,Fun,Party
1,5f00ec86304f7322eb8dfa41,2,Unlock,Door,Enter,Dreams,Time and space,Possibilites,Infinite,Time,...,Transparent,Glass,Shattered,Repair,Meaningful,Bond,Glue,All together,Matters,Wonders
2,5de27ced22383629b807cc70,3,Hole,Ground,Hog,Pig,Pork,Sald,Dressing,Clothes,...,Sunshine,Wamrth,Blanket,Heavy,Body,Muscle,Protein,Shake,Dance,Jump


In [5]:
seed_words = ['key', 'money', 'friend'] # 정해진 시드 단어들을 배열에 저장
target_words = ['key', 'money', 'friend'] # seed_words와 동일
n_respond_words = 30 # 하나의 시드당 30개의 단어 응답
n_subject = len(pilot_data) # 58명의 피험자

In [6]:
# wmd_seed_target 컬럼 미리 생성(빈 값)
for target_word in seed_words:
    for seed_word in seed_words:
        column_name = f'wmd_{seed_word}_{target_word}'
        pilot_data[column_name] = 0

    column_name = f'wmd_{target_word}'
    pilot_data[column_name] = np.nan

pilot_data.columns

Index(['Prolific_ID', 'subject', 'key1', 'key2', 'key3', 'key4', 'key5',
       'key6', 'key7', 'key8',
       ...
       'wmd_friend_key', 'wmd_key', 'wmd_key_money', 'wmd_money_money',
       'wmd_friend_money', 'wmd_money', 'wmd_key_friend', 'wmd_money_friend',
       'wmd_friend_friend', 'wmd_friend'],
      dtype='object', length=104)

In [7]:
# 응답 단어 컬럼 목록 (예: 'key1', 'key2', ... , 'key30', 'money1', 'money2', ... , 'money30', 'friend1', 'friend2', ... , 'friend30')
word_columns = [f'{seed_word}{i}' for seed_word in seed_words for i in range(1, n_respond_words + 1)]
# word_columns

In [8]:
for i_subject in range(n_subject):
    # print('subject: ', i_subject)

    for target_word in target_words: # key, money, friend
        for seed_word in seed_words: # key, money, friend
            # 각 피험자의 응답 문서를 30개씩 단어 리스트로 반환
            seed_response_words = [pilot_data.iloc[i_subject][column] for column in word_columns if column.startswith(seed_word)]
            # nan값 걸러내기
            seed_response_words = [word for word in seed_response_words if not isinstance(word, float) or not np.isnan(word)] 

            if len(seed_response_words) > 0: # 애초에 값이 0인 피험자 데이터들은 건너뛰도록 함
                try:
                    # 응답 30개 단어 - target 단어의 WMD 계산
                    wmdistance_value = word2vec_model.wmdistance([target_word], seed_response_words)

                    # 계산한 WMD를 해당 테이블 위치에 저장
                    pilot_data.at[i_subject, f'wmd_{seed_word}_{target_word}'] = wmdistance_value
                    print(f'wmd_{seed_word}_{target_word}: {wmdistance_value}')
                except Exception as e:
                    print(f"An error occurred for subject {i_subject}: {str(e)}")
                    continue

        # 각 피험자의 응답 단어 90개 모두를 리스트로 반환
        total_response_words = [pilot_data.iloc[i_subject][column] for column in word_columns]
        # 응답 90개 단어 - target 단어의 WMD 계산
        total_wmdistance_value = word2vec_model.wmdistance([target_word], total_response_words)

        # 계산한 WMD를 해당 테이블 위치에 저장
        pilot_data.at[i_subject, f'wmd_{target_word}'] = total_wmdistance_value
        print(f'wmd_{target_word}: {total_wmdistance_value}')


/var/folders/99/w9lwt31s6gzbvts3vs5x6myr0000gn/T/ipykernel_24945/1147595627.py:17: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '1.3910585478548874' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  pilot_data.at[i_subject, f'wmd_{seed_word}_{target_word}'] = wmdistance_value
/var/folders/99/w9lwt31s6gzbvts3vs5x6myr0000gn/T/ipykernel_24945/1147595627.py:17: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '1.3942253176490893' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  pilot_data.at[i_subject, f'wmd_{seed_word}_{target_word}'] = wmdistance_value
/var/folders/99/w9lwt31s6gzbvts3vs5x6myr0000gn/T/ipykernel_24945/1147595627.py:17: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '1.3915910726066376' has

wmd_key_key: 1.3910585478548874
wmd_money_key: 1.3942253176490893
wmd_friend_key: 1.3915910726066376
wmd_key: 1.3923828126406326
wmd_key_money: 1.3845510880725866
wmd_money_money: 1.3491492930807292
wmd_friend_money: 1.383315853364714
wmd_money: 1.3711802312655899
wmd_key_friend: 1.3851616851166642
wmd_money_friend: 1.3744623151847684
wmd_friend_friend: 1.3459576823925368
wmd_friend: 1.369323991529271
wmd_key_key: 1.3633910786884351
wmd_money_key: 1.3708503212617251
wmd_friend_key: 1.364012203913959
wmd_key: 1.3662171178677116
wmd_key_money: 1.3726648953986884
wmd_money_money: 1.3770626622949786
wmd_friend_money: 1.3831599049968222
wmd_money: 1.378247524967348
wmd_key_friend: 1.3643633157840414
wmd_money_friend: 1.365838365079093
wmd_friend_friend: 1.3776102556984484
wmd_friend: 1.3699968163065634
wmd_key_key: 1.3857931180890057
wmd_money_key: 1.3809168682137871
wmd_friend_key: 1.369404984650579
wmd_key: 1.378907312952833
wmd_key_money: 1.375483996226461
wmd_money_money: 1.386650515469

In [9]:
pilot_data[0:3]

,Prolific_ID,subject,key1,key2,key3,key4,key5,key6,key7,key8,...,wmd_friend_key,wmd_key,wmd_key_money,wmd_money_money,wmd_friend_money,wmd_money,wmd_key_friend,wmd_money_friend,wmd_friend_friend,wmd_friend
0,5d53bffa147a7d00015aae5a,1,Door,Gate,Outside,Grass,Itchy,Rash,Chicken pox,Shingles,...,1.391591,1.392383,1.384551,1.349149,1.383316,1.371180,1.385162,1.374462,1.345958,1.369324
1,5f00ec86304f7322eb8dfa41,2,Unlock,Door,Enter,Dreams,Time and space,Possibilites,Infinite,Time,...,1.364012,1.366217,1.372665,1.377063,1.383160,1.378248,1.364363,1.365838,1.377610,1.369997
2,5de27ced22383629b807cc70,3,Hole,Ground,Hog,Pig,Pork,Sald,Dressing,Clothes,...,1.369405,1.378907,1.375484,1.386651,1.355898,1.372919,1.357167,1.362013,1.394552,1.370782


In [11]:
# 단어 있는 버전 csv 저장
pilot_data.to_csv(processed_data_dir + 'distance_with_words.csv', index=None)

# 단어 컬럼들 드롭
drop_columns = pilot_data.columns[1:91]
pilot_data = pilot_data.drop(drop_columns, axis='columns')

# 단어 없이 coherence만 있는 버전 csv 저장
pilot_data.to_csv(processed_data_dir + 'distance.csv', index=None)